In [2]:
print("hello world")

hello world


In [3]:
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

model = SentenceTransformer("all-MiniLM-L6-v2")

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

In [9]:
query_vector = model.encode("How do I run Kafka?")
#results = vs_index.search(query_vector, num_results=5)
results= vs_index.search(
    query_vector,
    filter_dict={'course':'llm-zoomcamp'},
    num_results=5
)

In [10]:
results

[{'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'WSL2: ResponseError: model requires more system memory (X.X GiB) than is available (Y.Y GiB). My system has more than X.X GiB.',
  'answer': 'Your WSL2 is set to use Y.Y GiB, not all your computer memory. To allocate more RAM, follow these steps:\n\n1. Create a `.wslconfig` file under your Windows user profile directory: `C:\\Users\\YourUsername\\.wslconfig`.\n\n2. Include the desired RAM allocation in the file:\n\n   ```ini\n   [wsl2]\n   memory=8GB\n   ```\n\n3. Restart WSL using the command:\n\n   ```bash\n   wsl --shutdown\n   ```\n\n4. Run the `free` command in WSL to verify the changes.\n\nFor more details, read [this article](https://www.aleksandrhovhannisyan.com/blog/limiting-memory-usage-in-wsl-2/).',
  'doc_id': 'f3dd94f323'}]

In [11]:
from rag_helper import RAGBase
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [12]:
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )


In [13]:
vector_assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=openai_client,
)

In [14]:
vector_assistant.rag("the program has already begun, can I still sign up?")


'Yes, you can still join. If you want to receive a certificate, make sure you submit your project while submissions are still open.'

In [ ]:
vs_index.close()
